<a href="https://colab.research.google.com/github/dimasjanotama/data-science-2026/blob/main/Pertemuan3_%5BDimas_Janotama%5D_%5B240401010264%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nama : Dimas Prih P J<br>
NIM : 240401010264<br>
Kelas : IF403<br>

In [46]:
import requests, pandas as pd, numpy as np
from pandas import json_normalize
from scipy.stats.mstats import winsorize

In [17]:
#3. Muat Dataset (URL)
file_id = "1LfQWProB0VjWN5q8bKuRIgn-stULfIRo"
url = f"https://drive.google.com/uc?export=download&id={file_id}"
df = pd.read_csv(url)
# print(df.head())

In [34]:
#4. Eksplorasi Awal
print(df.info())
print(df.describe())
print('\n\nShape awal:', df.shape)
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       130 non-null    float64
 2   harga_juta    130 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         130 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB
None
               id      luas_m2    harga_juta       kamar  tahun_bangun
count  130.000000   130.000000  1.300000e+02  130.000000    130.000000
mean    65.500000   257.405385  7.699047e+05    3.476923   2062.638462
std     37.671829   821.951924  8.770521e+06    1.712767    701.684043
min      1.000000   -50.000000 -5.000000e+02    1.000000   1890.000000
25%     33.250000   101.600000  3.805000e+02    2.000000   1991.250000
50%     65.50000

In [36]:
#5. Hapus baris duplikat
#Deteksi: berapa baris yang duplikat?
n_dup = df.duplicated().sum()
print(f'{n_dup} baris duplikat dari {len(df)} total')

#Hapus baris duplikat
df.drop_duplicates(inplace=True)

0 baris duplikat dari 130 total


In [21]:
#6. Melakukan Normalisasi String
#Normalisasi kolom kota: menghapus spasi di awal/akhir dan membuat huruf kapital setiap kata
df['kota'] = df['kota'].astype(str).str.strip().str.title()

#Normalisasi kolom kondisi: mengubah semua teks menjadi huruf kecil
df['kondisi'] = df['kondisi'].astype(str).str.lower()

#Menampilkan hasil pengecekan
print(df[['kota', 'kondisi']].head())

    kota kondisi
0  Jogja    baik
1  Medan   bagus
2  Depok    baik
3    Ygy    baik
4  Medan  sedang


In [23]:
#7. Imputasi missing values
#Imputasi kolom Numerik dengan Median
#Identifikasi kolom numerik (float atau int)
numeric_cols = df.select_dtypes(include=['number']).columns

for col in numeric_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

#Imputasi kolom Kategorik dengan Modus
#Identifikasi kolom kategorik (object atau category)
categorical_cols = df.select_dtypes(include=['object', 'category']).columns

for col in categorical_cols:
    #mode() mengembalikan Series, ambil nilai pertama [0]
    mode_val = df[col].mode()[0]
    df[col] = df[col].fillna(mode_val)

print("Done Imputasi")

Done Imputasi


In [33]:
#8. Menangani Outlier dengan IQR Fence
#Kolom harga_juta
def deteksi_outlier_iqr(df, kolom):
    Q1  = df['harga_juta'].quantile(0.25)
    Q3  = df['harga_juta'].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df['harga_juta']<lower)|(df['harga_juta']>upper)]
    return lower, upper, outliers

lower, upper, out_df = deteksi_outlier_iqr(df, 'harga_juta')
print('Kolom harga_juta')
print(f'Outlier: {len(out_df)} baris')
print(f'Batas: [{lower:.0f}, {upper:.0f}] \n')

#Kolom harga_juta
def deteksi_outlier_iqr(df, kolom):
    Q1  = df['luas_m2'].quantile(0.25)
    Q3  = df['luas_m2'].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df['luas_m2']<lower)|(df['luas_m2']>upper)]
    return lower, upper, outliers

lower, upper, out_df = deteksi_outlier_iqr(df, 'luas_m2')
print('Kolom luas_m2')
print(f'Outlier: {len(out_df)} baris')
print(f'Batas: [{lower:.0f}, {upper:.0f}] \n\nOutlier sudah ditangani')

Kolom harga_juta
Outlier: 3 baris
Batas: [-423, 1719] 

Kolom luas_m2
Outlier: 1 baris
Batas: [-145, 513] 

Outlier sudah ditangani


In [35]:
#9. Validasi Akhir
null_count = df.isnull().sum().sum()
duplicated_count = df.duplicated().sum()

print(f"Total Missing Values: {null_count}")
print(f"Total Duplikat: {duplicated_count}")

if null_count == 0 and duplicated_count == 0:
    print("Data sudah bersih")
else:
    print("Masih ada data yang perlu diperbaiki.")

Total Missing Values: 0
Total Duplikat: 0
Validasi sukses: Data sudah bersih!


In [39]:
#10. Ekspor data bersih
df.to_csv('housing_clean.csv', index=False)
print('Dataset bersih tersimpan!')

Dataset bersih tersimpan!


In [54]:
#Akses API JSONPlaceholder
# 1 Contoh: JSONPlaceholder (gratis, tanpa API key)
URL = "https://jsonplaceholder.typicode.com/users"
response = requests.get(URL, timeout=10)

# 2 Selalu cek status code terlebih dahulu
if response.status_code == 200:
    data = response.json()
    df = json_normalize(data, sep='_')
    # print(df.head())
    print(df[['id','name','email','address_city']])
else:
    print(f'Error: {response.status_code}')

# 3 API dengan parameter (query string)
params = {'userId': 1} # filter by user
posts = requests.get("https://jsonplaceholder.typicode.com/posts",
                      params=params).json()
df_posts = pd.DataFrame(posts)

   id                      name                      email    address_city
0   1             Leanne Graham          Sincere@april.biz     Gwenborough
1   2              Ervin Howell          Shanna@melissa.tv     Wisokyburgh
2   3          Clementine Bauch         Nathan@yesenia.net   McKenziehaven
3   4          Patricia Lebsack  Julianne.OConner@kory.org     South Elvis
4   5          Chelsey Dietrich   Lucio_Hettinger@annie.ca      Roscoeview
5   6      Mrs. Dennis Schulist    Karley_Dach@jasper.info   South Christy
6   7           Kurtis Weissnat     Telly.Hoeger@billy.biz       Howemouth
7   8  Nicholas Runolfsdottir V       Sherwood@rosamond.me       Aliyaview
8   9           Glenna Reichert    Chaim_McDermott@dana.io  Bartholomebury
9  10        Clementina DuBuque     Rey.Padberg@karina.biz     Lebsackbury
